In [0]:
for stream in spark.streams.active:
    print(f"A parar stream: {stream.id}")
    stream.stop()

import time
time.sleep(5)

spark.sql("DROP TABLE IF EXISTS bus_telemetry_raw")
spark.sql("DROP TABLE IF EXISTS charger_telemetry_raw")
spark.sql("DROP VIEW IF EXISTS bus_telemetry")
spark.sql("DROP VIEW IF EXISTS charger_telemetry")
print("Tabelas e views eliminadas")

dbutils.fs.rm("dbfs:/Workspace/checkpoints/bus_telemetry", True)
dbutils.fs.rm("dbfs:/Workspace/checkpoints/charger_telemetry", True)
print("Checkpoints eliminados")

print("\nRESET COMPLETO! Agora executa as celulas de stream novamente")

In [0]:
try:
    spark.sql("SELECT COUNT(*) FROM bus_telemetry").show()
    print("tabela ainda existe")
except:
    print("tabela eliminada")

# STCP - Análise de Cenário GTFS Real

Análise de autocarros elétricos em rotas reais do Porto:
- BUS-001: Rota 200 (Bolhão - Castelo Queijo)
- BUS-002: Rota 201 (Aliados - Viso)
- BUS-003: Rota 202 (Aliados - Passeio Alegre)

Última atualização: Março 2026

## Passo 1: Configuração Event Hub

Ligação ao Azure Event Hub para receber telemetria em tempo real.

In [0]:
# Configuração do Event Hub
# Estes parâmetros estabelecem a ligação ao Azure Event Hub

# Nome do namespace onde está o Event Hub
eh_namespace = "ehns-stcp-bus"

# Nome do Event Hub específico que contém a telemetria
eh_name = "stcp-telemetry"

# Nome da política de acesso (Manage permite ler e escrever)
eh_key_name = "RootManageSharedAccessKey"

# Chave de acesso (Primary Key da política Manage)
eh_key = "ZdvXUobLzuB4r4I+9X1qBVOkIHLZ62gIw+AEhPRpFiY="

# String de autenticação SASL para Kafka
# Formato específico para Event Hub: usa a connection string como password
eh_sasl = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="Endpoint=sb://{eh_namespace}.servicebus.windows.net/;SharedAccessKeyName={eh_key_name};SharedAccessKey={eh_key};EntityPath={eh_name}";'

print("Configuração concluída com sucesso")

## Passo 3: Ler Dados do Event Hub

Criar streaming DataFrame que lê continuamente do Event Hub.

**Configuração**:
- `startingOffsets: "latest"` - Apenas dados NOVOS
- `startingOffsets: "earliest"` - TODO o histórico + novos

Configurado para **"latest"**.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Criar streaming DataFrame
df_stream = spark.readStream.format("kafka").option("kafka.bootstrap.servers", f"{eh_namespace}.servicebus.windows.net:9093").option("subscribe", eh_name).option("kafka.sasl.mechanism", "PLAIN").option("kafka.security.protocol", "SASL_SSL").option("kafka.sasl.jaas.config", eh_sasl).option("kafka.request.timeout.ms", "60000").option("kafka.session.timeout.ms", "30000").option("failOnDataLoss", "false").option("startingOffsets", "earliest").load()

print("Stream configurado!")
print("Colunas disponiveis:", df_stream.columns)

## Passo 4: Parse do JSON e Separacao de Dados
**Parse JSON** = Transformar esse texto em **colunas**:
Processar JSON e separar em duas streams:
- Autocarros: batteryKwh, batteryLevel, batteryTemperature, latitude, longitude, state
- Estacoes: chargerTemperature, currentEfficiency, currentPower, energyDelivered, state

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Schema COMPLETO do JSON (com telemetry wrapper)
full_schema = StructType([
    StructField("deviceId", StringType(), True),
    StructField("telemetry", StructType([
        StructField("batteryKwh", DoubleType(), True),
        StructField("batteryLevel", DoubleType(), True),
        StructField("batteryTemperature", DoubleType(), True),
        StructField("latitude", DoubleType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("state", StringType(), True),
        StructField("instantConsumption", DoubleType(), True),
        StructField("remainingRange", DoubleType(), True),
        StructField("distanceTraveled", DoubleType(), True),
        StructField("chargerTemperature", DoubleType(), True),
        StructField("currentEfficiency", DoubleType(), True),
        StructField("currentPower", DoubleType(), True),
        StructField("energyDelivered", DoubleType(), True),
        StructField("connectedBus", StringType(), True),
        StructField("totalEnergyDelivered", DoubleType(), True)
    ]), True)
])

# Parse JSON completo
df_parsed = df_stream.select(
    from_json(col("value").cast("string"), full_schema).alias("data"),
    col("timestamp").alias("enqueued_time")
).withColumn("event_time", current_timestamp())

# Extrair dados dos autocarros
df_bus_stream = df_parsed.filter(col("data.deviceId").rlike("BUS-")).select(
    col("event_time"),
    col("data.deviceId").alias("device_id"),
    col("enqueued_time"),
    col("data.telemetry.batteryKwh").alias("batteryKwh"),
    col("data.telemetry.batteryLevel").alias("batteryLevel"),
    col("data.telemetry.batteryTemperature").alias("batteryTemperature"),
    col("data.telemetry.latitude").alias("latitude"),
    col("data.telemetry.longitude").alias("longitude"),
    col("data.telemetry.state").alias("state"),
    col("data.telemetry.instantConsumption").alias("instantConsumption"),
    col("data.telemetry.remainingRange").alias("remainingRange"),
    col("data.telemetry.distanceTraveled").alias("distanceTraveled")
)

# Extrair dados das estacoes
df_charger_stream = df_parsed.filter(col("data.deviceId").rlike("CS-")).select(
    col("event_time"),
    col("data.deviceId").alias("device_id"),
    col("enqueued_time"),
    col("data.telemetry.chargerTemperature").alias("chargerTemperature"),
    col("data.telemetry.currentEfficiency").alias("currentEfficiency"),
    col("data.telemetry.currentPower").alias("currentPower"),
    col("data.telemetry.energyDelivered").alias("energyDelivered"),
    col("data.telemetry.state").alias("state"),
    col("data.telemetry.connectedBus").alias("connectedBus"),
    col("data.telemetry.totalEnergyDelivered").alias("totalEnergyDelivered"),
    col("data.telemetry.latitude").alias("latitude"),
    col("data.telemetry.longitude").alias("longitude")
)

print("Schema actualizado com distanceTraveled!")

## Passo 5: Gravar Streams em Delta Tables

Criar tabelas Delta para armazenar telemetria:
- bus_telemetry
- charger_telemetry

In [0]:
# Gravar stream de AUTOCARROS na tabela raw
query_bus = df_bus_stream.writeStream.format("delta").outputMode("append").option("checkpointLocation", "dbfs:/Workspace/checkpoints/bus_telemetry").table("bus_telemetry_raw")

# Gravar stream de ESTACOES na tabela raw
query_charger = df_charger_stream.writeStream.format("delta").outputMode("append").option("checkpointLocation", "dbfs:/Workspace/checkpoints/charger_telemetry").table("charger_telemetry_raw")

print("Streams a gravar em Delta Tables!")
print("Aguarda 30-60 segundos para dados chegarem...")

In [0]:
# Criar views com os nomes originais que filtram apenas dados dos ultimos 2 minutos
# Assim todas as queries continuam a usar bus_telemetry e charger_telemetry
# mas mostram sempre apenas dados recentes, evitando dados antigos do Event Hub

spark.sql("""
    CREATE OR REPLACE VIEW bus_telemetry AS
    SELECT * FROM bus_telemetry_raw
    WHERE event_time >= current_timestamp() - INTERVAL 2 MINUTES
""")

spark.sql("""
    CREATE OR REPLACE VIEW charger_telemetry AS
    SELECT * FROM charger_telemetry_raw
    WHERE event_time >= current_timestamp() - INTERVAL 2 MINUTES
""")

print("Views criadas com filtro de 2 minutos")
print("Todas as queries continuam a usar os mesmos nomes de tabela")

In [0]:
# Ver se streams estão ativos
print("=== STREAMS ATIVOS ===")
for stream in spark.streams.active:
    print(f"Stream: {stream.name} | Status: {stream.status}")
print(f"Total: {len(spark.streams.active)} streams")

## Passo 6: Verificar Dados

Aguardar 60 segundos e verificar se dados estão a chegar.

## Visualização de Dados - Autocarros

Todas as características dos autocarros em tempo real.

In [0]:
print("=== AUTOCARROS - TODAS AS CARACTERÍSTICAS ===")

display(spark.sql("""
    SELECT 
        device_id,
        state,
        round(batteryLevel, 3) as bateria_percentagem,
        round(batteryKwh, 2) as bateria_kwh,
        round(batteryTemperature, 1) as temperatura_celsius,
        round(latitude, 6) as latitude,
        round(longitude, 6) as longitude,
        round(distanceTraveled, 0) as distancia_percorrida_metros,
        date_format(event_time, 'dd-MM-yyyy HH:mm:ss') as data_hora
    FROM bus_telemetry
    ORDER BY event_time DESC
    LIMIT 20
"""))

In [0]:
print("=== ESTAÇÕES DE CARREGAMENTO - TODAS AS CARACTERÍSTICAS ===")

display(spark.sql("""
    SELECT 
        device_id,
        state,
        currentPower,
        currentEfficiency,
        chargerTemperature,
        energyDelivered,
        totalEnergyDelivered,
        connectedBus,
        latitude,
        longitude,
        date_format(event_time, 'dd-MM-yyyy HH:mm:ss') as data_hora
    FROM charger_telemetry
    ORDER BY event_time DESC
    LIMIT 20
"""))

Horario de cada autocarro, tendo em conta a linha

In [0]:
from pyspark.sql import Row

paragens = [
    # LINHA 200 - BUS-001 - Bolhão - Castelo do Queijo
    Row(bus_id="BUS-001", route_id="200", stop_sequence=1,  stop_id="BLRB1",  stop_name="Bolhão",                arrival_time="06:00", distance_meters=0.0,     stop_lat=41.151868, stop_lon=-8.607115),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=2,  stop_id="MCBL",   stop_name="Mercado Do Bolhão",    arrival_time="06:01", distance_meters=326.09,  stop_lat=41.149509, stop_lon=-8.607564),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=3,  stop_id="PRDJ",   stop_name="Pr. D. João I",        arrival_time="06:02", distance_meters=621.59,  stop_lat=41.147581, stop_lon=-8.609126),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=4,  stop_id="PRFL",   stop_name="Pr.filipa De Lencastre", arrival_time="06:03", distance_meters=944.8, stop_lat=41.148261, stop_lon=-8.612768),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=5,  stop_id="GGF",    stop_name="Guil. G. Fernandes",   arrival_time="06:04", distance_meters=1174.83, stop_lat=41.147560, stop_lon=-8.614767),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=6,  stop_id="CMO",    stop_name="Carmo",                arrival_time="06:05", distance_meters=1387.32, stop_lat=41.147223, stop_lon=-8.616926),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=7,  stop_id="HSA5",   stop_name="Hosp. St. António",    arrival_time="06:07", distance_meters=1962.62, stop_lat=41.147766, stop_lon=-8.622450),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=8,  stop_id="PAL2",   stop_name="Palácio",              arrival_time="06:08", distance_meters=2288.64, stop_lat=41.149145, stop_lon=-8.625447),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=9,  stop_id="PRG4",   stop_name="Pr. Da Galiza",        arrival_time="06:10", distance_meters=2770.22, stop_lat=41.152531, stop_lon=-8.627041),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=10, stop_id="JM1",    stop_name="Junta Massarelos",     arrival_time="06:10", distance_meters=3129.94, stop_lat=41.152694, stop_lon=-8.631221),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=11, stop_id="GGT1",   stop_name="Gólgota",              arrival_time="06:11", distance_meters=3343.19, stop_lat=41.152639, stop_lon=-8.633694),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=12, stop_id="PLNT1",  stop_name="Planetário",           arrival_time="06:12", distance_meters=3746.09, stop_lat=41.153094, stop_lon=-8.638289),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=13, stop_id="FCUP1",  stop_name="Faculdade De Ciências", arrival_time="06:13", distance_meters=4013.42, stop_lat=41.153838, stop_lon=-8.641309),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=14, stop_id="JB1",    stop_name="Jardim Botânico",      arrival_time="06:13", distance_meters=4271.82, stop_lat=41.154528, stop_lon=-8.644222),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=15, stop_id="LRD1",   stop_name="Lordelo",              arrival_time="06:15", distance_meters=4679.94, stop_lat=41.154389, stop_lon=-8.649028),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=16, stop_id="PLM1",   stop_name="Palmeiras",            arrival_time="06:15", distance_meters=5046.91, stop_lat=41.152356, stop_lon=-8.652107),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=17, stop_id="FLUN1",  stop_name="Fluvial (norte)",      arrival_time="06:16", distance_meters=5319.24, stop_lat=41.152139, stop_lon=-8.655194),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=18, stop_id="PG1",    stop_name="Paulo Da Gama",        arrival_time="06:17", distance_meters=5656.47, stop_lat=41.150856, stop_lon=-8.658707),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=19, stop_id="PT3",    stop_name="Pasteleira",           arrival_time="06:17", distance_meters=5895.45, stop_lat=41.150486, stop_lon=-8.661488),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=20, stop_id="TRR3",   stop_name="Torres",               arrival_time="06:18", distance_meters=6163.89, stop_lat=41.150749, stop_lon=-8.664619),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=21, stop_id="PCL1",   stop_name="Padre Luis Cabral",    arrival_time="06:19", distance_meters=6481.06, stop_lat=41.151806, stop_lon=-8.668056),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=22, stop_id="UC1",    stop_name="Univ. Católica",       arrival_time="06:20", distance_meters=6793.05, stop_lat=41.153538, stop_lon=-8.670916),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=23, stop_id="MFZ1",   stop_name="Mercado Da Foz",       arrival_time="06:21", distance_meters=7199.66, stop_lat=41.155048, stop_lon=-8.673343),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=24, stop_id="LIEG1",  stop_name="Pr. De Liége",         arrival_time="06:22", distance_meters=7692.57, stop_lat=41.155089, stop_lon=-8.677168),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=25, stop_id="CRTO3",  stop_name="Crasto",               arrival_time="06:23", distance_meters=8098.83, stop_lat=41.157713, stop_lon=-8.679766),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=26, stop_id="MLH5",   stop_name="Molhe",                arrival_time="06:23", distance_meters=8389.4,  stop_lat=41.159880, stop_lon=-8.681660),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=27, stop_id="FCH1",   stop_name="Funchal",              arrival_time="06:24", distance_meters=8789.98, stop_lat=41.163190, stop_lon=-8.683490),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=28, stop_id="HMLM3",  stop_name="Homem Do Leme",        arrival_time="06:25", distance_meters=9077.4,  stop_lat=41.163454, stop_lon=-8.685965),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=29, stop_id="TIM1",   stop_name="Timor",                arrival_time="06:25", distance_meters=9338.03, stop_lat=41.165587, stop_lon=-8.687170),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=30, stop_id="CQ10",   stop_name="Castelo Do Queijo",    arrival_time="06:27", distance_meters=9853.9,  stop_lat=41.167422, stop_lon=-8.689127),

    # LINHA 201 - BUS-002 - Aliados - Viso
    Row(bus_id="BUS-002", route_id="201", stop_sequence=1,  stop_id="AL1",    stop_name="Av. Aliados",           arrival_time="07:38", distance_meters=0.0,     stop_lat=41.147208, stop_lon=-8.610983),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=2,  stop_id="TRD6",   stop_name="Trindade",              arrival_time="07:40", distance_meters=696.96,  stop_lat=41.151033, stop_lon=-8.610854),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=3,  stop_id="PRFL",   stop_name="Pr.filipa De Lencastre", arrival_time="07:41", distance_meters=1194.03, stop_lat=41.148261, stop_lon=-8.612768),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=4,  stop_id="GGF",    stop_name="Guil. G. Fernandes",    arrival_time="07:42", distance_meters=1423.58, stop_lat=41.147560, stop_lon=-8.614767),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=5,  stop_id="CMO",    stop_name="Carmo",                 arrival_time="07:43", distance_meters=1631.97, stop_lat=41.147223, stop_lon=-8.616926),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=6,  stop_id="HSA5",   stop_name="Hosp. St. António",     arrival_time="07:45", distance_meters=2206.8,  stop_lat=41.147766, stop_lon=-8.622450),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=7,  stop_id="PAL3",   stop_name="Palácio",               arrival_time="07:46", distance_meters=2568.12, stop_lat=41.149512, stop_lon=-8.625532),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=8,  stop_id="PRG1",   stop_name="Pr. Da Galiza",         arrival_time="07:48", distance_meters=3018.77, stop_lat=41.153444, stop_lon=-8.626333),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=9,  stop_id="BS1",    stop_name="Boavista - B.sucesso",  arrival_time="07:49", distance_meters=3388.02, stop_lat=41.156403, stop_lon=-8.627930),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=10, stop_id="BCM1",   stop_name="Boavista-casa Da Música", arrival_time="07:52", distance_meters=3787.25, stop_lat=41.158950, stop_lon=-8.629420),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=11, stop_id="AGM1",   stop_name="Agramonte",             arrival_time="07:54", distance_meters=4277.59, stop_lat=41.158948, stop_lon=-8.634595),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=12, stop_id="ACRD1",  stop_name="António Cardoso",       arrival_time="07:56", distance_meters=4735.65, stop_lat=41.159843, stop_lon=-8.639918),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=13, stop_id="BSS1",   stop_name="Bessa",                 arrival_time="07:58", distance_meters=5091.66, stop_lat=41.160520, stop_lon=-8.644060),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=14, stop_id="FOCO1",  stop_name="Foco",                  arrival_time="07:59", distance_meters=5384.38, stop_lat=41.161090, stop_lon=-8.647460),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=15, stop_id="PINM1",  stop_name="Pinheiro Manso",        arrival_time="08:01", distance_meters=5770.99, stop_lat=41.161845, stop_lon=-8.651927),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=16, stop_id="SRV3",   stop_name="Serralves",             arrival_time="08:04", distance_meters=6288.65, stop_lat=41.162848, stop_lon=-8.657954),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=17, stop_id="PNV2",   stop_name="Paulo Novais",          arrival_time="08:05", distance_meters=6470.52, stop_lat=41.163200, stop_lon=-8.660057),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=18, stop_id="FTM1",   stop_name="Fonte Da Moura",        arrival_time="08:07", distance_meters=6821.25, stop_lat=41.164505, stop_lon=-8.662144),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=19, stop_id="BRV1",   stop_name="Bairro Vilarinha",      arrival_time="08:08", distance_meters=7161.8,  stop_lat=41.167065, stop_lon=-8.659990),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=20, stop_id="LDD1",   stop_name="Lidador",               arrival_time="08:09", distance_meters=7524.24, stop_lat=41.169798, stop_lon=-8.657682),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=21, stop_id="PRO1",   stop_name="Pereiró",               arrival_time="08:10", distance_meters=7865.77, stop_lat=41.172368, stop_lon=-8.655497),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=22, stop_id="EZC1",   stop_name="Ezequiel Campos",       arrival_time="08:11", distance_meters=8140.75, stop_lat=41.174449, stop_lon=-8.653793),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=23, stop_id="MPAZ5",  stop_name="Man.p.azevedo 3",       arrival_time="08:13", distance_meters=8522.05, stop_lat=41.176989, stop_lon=-8.651836),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=24, stop_id="IMTT1",  stop_name="Imtt",                  arrival_time="08:14", distance_meters=8960.39, stop_lat=41.178485, stop_lon=-8.648801),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=25, stop_id="VIS5",   stop_name="Viso (metro)",          arrival_time="08:16", distance_meters=9306.3,  stop_lat=41.177361, stop_lon=-8.646500),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=26, stop_id="VIS3",   stop_name="Viso",                  arrival_time="08:18", distance_meters=9777.63, stop_lat=41.177840, stop_lon=-8.642280),

    # LINHA 202 - BUS-003 - Aliados - Passeio Alegre
    Row(bus_id="BUS-003", route_id="202", stop_sequence=1,  stop_id="SJB1",   stop_name="S. João De Brito",      arrival_time="06:43", distance_meters=0.0,     stop_lat=41.166494, stop_lon=-8.650585),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=2,  stop_id="ABM1",   stop_name="Alberto Macedo",        arrival_time="06:44", distance_meters=336.7,   stop_lat=41.167151, stop_lon=-8.654485),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=3,  stop_id="BRP1",   stop_name="Bairro Da Previdência", arrival_time="06:45", distance_meters=695.22,  stop_lat=41.165783, stop_lon=-8.657654),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=4,  stop_id="BRV3",   stop_name="Bairro Da Vilarinha",   arrival_time="06:45", distance_meters=925.14,  stop_lat=41.166094, stop_lon=-8.659405),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=5,  stop_id="FTM2",   stop_name="Fonte Da Moura",        arrival_time="06:47", distance_meters=1354.35, stop_lat=41.164191, stop_lon=-8.662646),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=6,  stop_id="ABVT1",  stop_name="Av.da Boavista",        arrival_time="06:47", distance_meters=1525.1,  stop_lat=41.162823, stop_lon=-8.663403),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=7,  stop_id="LGO4",   stop_name="Liceu Garcia Orta",     arrival_time="06:48", distance_meters=1877.3,  stop_lat=41.161528, stop_lon=-8.666778),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=8,  stop_id="RCRT2",  stop_name="R.crasto",              arrival_time="06:49", distance_meters=2350.57, stop_lat=41.162000, stop_lon=-8.671694),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=9,  stop_id="LNEV2",  stop_name="Lgo.nevogilde",         arrival_time="06:50", distance_meters=2757.63, stop_lat=41.162833, stop_lon=-8.676306),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=10, stop_id="NEVG",   stop_name="Nevogilde",             arrival_time="06:51", distance_meters=3082.04, stop_lat=41.161511, stop_lon=-8.679230),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=11, stop_id="MLH3",   stop_name="Molhe",                 arrival_time="06:52", distance_meters=3483.16, stop_lat=41.160158, stop_lon=-8.683185),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=12, stop_id="JNN2",   stop_name="Jacinto Nunes",         arrival_time="06:53", distance_meters=3750.03, stop_lat=41.158107, stop_lon=-8.681560),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=13, stop_id="CRTO4",  stop_name="Crasto",                arrival_time="06:53", distance_meters=3936.85, stop_lat=41.156600, stop_lon=-8.680260),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=14, stop_id="PING3",  stop_name="Praia Dos Ingleses",    arrival_time="06:54", distance_meters=4490.81, stop_lat=41.152767, stop_lon=-8.678563),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=15, stop_id="PORI2",  stop_name="Praia Do Ourigo",       arrival_time="06:56", distance_meters=4937.66, stop_lat=41.149376, stop_lon=-8.675942),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=16, stop_id="PASS1",  stop_name="Passeio Alegre",        arrival_time="06:57", distance_meters=5313.2,  stop_lat=41.148820, stop_lon=-8.672750),
]

df_paragens = spark.createDataFrame(paragens)
df_paragens.write.format("delta").mode("overwrite").saveAsTable("route_stops")
print(f"Tabela route_stops criada com {df_paragens.count()} paragens")

In [0]:
display(spark.sql("""
    SELECT device_id, round(batteryLevel, 4) as bateria, distanceTraveled, event_time
    FROM bus_telemetry
    WHERE device_id = 'BUS-001'
    ORDER BY event_time DESC
    LIMIT 10
"""))

## Análise de Autonomia por Rota

Cálculo do número de viagens completas que cada autocarro consegue realizar
partindo do depósito com bateria a **95% (SOC máximo operacional = 332.5 kWh)**,
até atingir o limite mínimo de **20% (reserva operacional = 70.0 kWh)**.

Cada viagem inclui variação aleatória:
- **Desvio de rota**: +2% a +12% sobre a distância base (tráfego, obras, etc.)
- **Consumo auxiliar por paragem**: 0.03 a 0.07 kWh (ar condicionado, abertura de portas, etc.)

In [0]:
# ============================================================
# ANÁLISE: Quantas viagens cada autocarro consegue completar
#          partindo do depósito com bateria a 95% (carga máxima)
# ============================================================

import random

# Parâmetros do CaetanoBus e.City Gold
BATTERY_CAPACITY_KWH   = 350.0   # Capacidade total da bateria
AVG_CONSUMPTION_KWH_KM = 1.2     # Consumo médio em kWh por km
SOC_DEPARTURE_PERCENT  = 95.0    # SOC máximo operacional (partida do depósito)
SOC_MIN_PERCENT        = 20.0    # SOC mínimo operacional (reserva de segurança)

BATTERY_DEPARTURE_KWH = (SOC_DEPARTURE_PERCENT / 100) * BATTERY_CAPACITY_KWH  # 332.5 kWh
SOC_MIN_KWH           = (SOC_MIN_PERCENT       / 100) * BATTERY_CAPACITY_KWH  # 70.0 kWh
USABLE_ENERGY_KWH     = BATTERY_DEPARTURE_KWH - SOC_MIN_KWH                   # 262.5 kWh

# Dados das rotas (distâncias reais da tabela route_stops)
route_data = {
    "BUS-001": {"route_name": "Rota 200 - Bolhão -> Castelo do Queijo",      "base_distance_km": 9.854, "num_stops": 30},
    "BUS-002": {"route_name": "Rota 201 - Aliados -> Viso",                   "base_distance_km": 9.778, "num_stops": 26},
    "BUS-003": {"route_name": "Rota 202 - S.Joao de Brito -> Passeio Alegre", "base_distance_km": 5.313, "num_stops": 16},
}

results = []
for device_id, route in route_data.items():
    base_dist  = route["base_distance_km"]
    num_stops  = route["num_stops"]
    route_name = route["route_name"]

    trips_done  = 0
    energy_left = USABLE_ENERGY_KWH
    trip_log    = []

    while energy_left > 0:
        # Variação aleatória: desvio de rota (+2% a +12%)
        detour_factor  = 1 + random.uniform(0.02, 0.12)
        effective_dist = base_dist * detour_factor

        # Variação aleatória: consumo auxiliar por paragem (0.03 a 0.07 kWh)
        aux_per_stop    = random.uniform(0.03, 0.07)
        aux_consumption = num_stops * aux_per_stop

        # Energia total da viagem = tracção + auxiliares
        trip_energy = (effective_dist * AVG_CONSUMPTION_KWH_KM) + aux_consumption

        if energy_left < trip_energy:
            break

        energy_left -= trip_energy
        trips_done  += 1
        trip_log.append(round(trip_energy, 2))

    # Definir alerta operacional
    if trips_done == 0:
        alert = "CARREGAR AGORA"
    elif trips_done == 1:
        alert = "CARREGAR ANTES DA PROXIMA VIAGEM"
    else:
        alert = "OK"

    results.append({
        "Autocarro":                      device_id,
        "Rota":                           route_name,
        "SOC Partida (%)": SOC_DEPARTURE_PERCENT,
        "Energia Util (kWh)":             round(USABLE_ENERGY_KWH, 1),
        "Viagens Possiveis":              trips_done,
        "Energia Media por Viagem (kWh)": round(sum(trip_log) / len(trip_log), 2) if trip_log else 0,
        "Energia Restante (kWh)":         round(energy_left, 2),
        "Alerta":                         alert,
    })

results_df = spark.createDataFrame(results)
display(results_df.orderBy("Viagens Possiveis"))